# Hybrid Modwheel + Confidence Readout Scheduling (Colab)

Notebook 11 combines two readout-side ideas:

```text
deterministic modwheel structure
+
confidence-aware query prioritization
```

Notebook 10 tested global confidence-aware scheduling. Notebook 11 asks whether modular lanes can provide structured coverage while confidence ranking prioritizes uncertain queries.

Core idea:

```text
many sparse test queries
    -> split into mod30 lanes
    -> rank low-confidence queries within each lane
    -> interleave lanes
    -> progressive readout / early stopping
```

Guardrail: this notebook evaluates classical readout scheduling behavior. It does **not** claim QOS improvement, quantum advantage, or model accuracy improvement.


In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these two lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)


## 1. Load 20news and train baseline classifier

In [ ]:
RANDOM_STATE = 9423
N_RANDOM_TRIALS = 30

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc",
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

texts = np.array(dataset.data, dtype=object)
y = np.array(dataset.target)
target_names = dataset.target_names

texts_train, texts_test, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer = TfidfVectorizer(max_features=12000, min_df=2, stop_words="english")
X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)

clf = LinearSVC(random_state=RANDOM_STATE, dual="auto")
t0 = time.perf_counter()
clf.fit(X_train, y_train)
train_time = time.perf_counter() - t0

full_pred = clf.predict(X_test)
full_accuracy = accuracy_score(y_test, full_pred)
full_balanced_accuracy = balanced_accuracy_score(y_test, full_pred)

baseline = {
    "n_train": X_train.shape[0],
    "n_test": X_test.shape[0],
    "n_features": X_train.shape[1],
    "train_time_seconds": train_time,
    "full_accuracy": full_accuracy,
    "full_balanced_accuracy": full_balanced_accuracy,
}
baseline


## 2. Confidence and metric helpers

In [ ]:
def decision_margin_confidence(model, X):
    scores = model.decision_function(X)
    if scores.ndim == 1:
        return np.abs(scores)
    sorted_scores = np.sort(scores, axis=1)
    top = sorted_scores[:, -1]
    second = sorted_scores[:, -2]
    return top - second


confidence = decision_margin_confidence(clf, X_test)
all_idx = np.arange(X_test.shape[0])


def class_balance_l1_shift(y_subset, y_reference):
    n_classes = len(np.unique(y_reference))
    subset_counts = np.bincount(y_subset, minlength=n_classes)
    ref_counts = np.bincount(y_reference, minlength=n_classes)
    subset_frac = subset_counts / subset_counts.sum()
    ref_frac = ref_counts / ref_counts.sum()
    return float(np.sum(np.abs(subset_frac - ref_frac)))


def lane_coverage_fraction(indices, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    residues = set(wheel.residues)
    covered = set([int(i % wheel.modulus) for i in indices if int(i % wheel.modulus) in residues])
    return len(covered) / len(residues)


def evaluate_indices(indices):
    pred = clf.predict(X_test[indices])
    yt = y_test[indices]
    return {
        "accuracy": accuracy_score(yt, pred),
        "balanced_accuracy": balanced_accuracy_score(yt, pred),
        "class_balance_l1_shift": class_balance_l1_shift(yt, y_test),
        "mean_confidence": float(np.mean(confidence[indices])),
        "median_confidence": float(np.median(confidence[indices])),
        "lane_coverage_fraction": lane_coverage_fraction(indices, "mod30"),
    }


confidence_summary = {
    "min": float(np.min(confidence)),
    "median": float(np.median(confidence)),
    "mean": float(np.mean(confidence)),
    "max": float(np.max(confidence)),
}
confidence_summary


## 3. Schedule constructors

In [ ]:
def wheel_query_indices(n_queries, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    residues = set(wheel.residues)
    return np.array([i for i in range(n_queries) if i % wheel.modulus in residues], dtype=int)


def random_schedule(n_keep, seed):
    rng = np.random.default_rng(seed)
    return np.array(rng.choice(all_idx, size=n_keep, replace=False), dtype=int)


def global_low_confidence_schedule(n_keep):
    return all_idx[np.argsort(confidence[all_idx])][:n_keep]


def global_high_confidence_schedule(n_keep):
    return all_idx[np.argsort(-confidence[all_idx])][:n_keep]


def mod_lanes(indices, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    residues = list(wheel.residues)
    lanes = {r: [] for r in residues}
    for i in indices:
        r = int(i % wheel.modulus)
        if r in lanes:
            lanes[r].append(int(i))
    return lanes


def interleave_lanes(lanes):
    output = []
    keys = list(lanes.keys())
    max_len = max(len(v) for v in lanes.values() if len(v) > 0)
    for j in range(max_len):
        for k in keys:
            if j < len(lanes[k]):
                output.append(lanes[k][j])
    return np.array(output, dtype=int)


def hybrid_lane_confidence_schedule(n_keep, wheel_name="mod30", confidence_order="low"):
    # Candidate pool is the wheel-admissible subset.
    base = wheel_query_indices(X_test.shape[0], wheel_name)
    lanes = mod_lanes(base, wheel_name=wheel_name)

    for r, values in lanes.items():
        arr = np.array(values, dtype=int)
        if len(arr) == 0:
            lanes[r] = []
            continue
        if confidence_order == "low":
            arr = arr[np.argsort(confidence[arr])]
        elif confidence_order == "high":
            arr = arr[np.argsort(-confidence[arr])]
        else:
            raise ValueError("confidence_order must be 'low' or 'high'")
        lanes[r] = arr.tolist()

    return interleave_lanes(lanes)[:n_keep]


def hybrid_lane_round_robin_schedule(n_keep, wheel_name="mod30"):
    base = wheel_query_indices(X_test.shape[0], wheel_name)
    lanes = mod_lanes(base, wheel_name=wheel_name)
    return interleave_lanes(lanes)[:n_keep]


mod30_idx = wheel_query_indices(X_test.shape[0], "mod30")
n_keep = len(mod30_idx)

schedules = {
    "mod30": mod30_idx,
    "global_low_conf": global_low_confidence_schedule(n_keep),
    "global_high_conf": global_high_confidence_schedule(n_keep),
    "hybrid_lane_low_conf": hybrid_lane_confidence_schedule(n_keep, "mod30", "low"),
    "hybrid_lane_high_conf": hybrid_lane_confidence_schedule(n_keep, "mod30", "high"),
    "hybrid_lane_round_robin": hybrid_lane_round_robin_schedule(n_keep, "mod30"),
}

{k: (len(v), lane_coverage_fraction(v)) for k, v in schedules.items()}


## 4. Progressive curve and early stopping helpers

In [ ]:
fractions = np.linspace(0.05, 1.0, 20)


def progressive_curve(schedule_indices, fractions, schedule_name, schedule_type, trial=-1):
    rows = []
    total_queries = X_test.shape[0]
    for frac in fractions:
        k = max(2, int(np.ceil(len(schedule_indices) * frac)))
        idx = schedule_indices[:k]
        t0 = time.perf_counter()
        metrics = evaluate_indices(idx)
        eval_time = time.perf_counter() - t0
        rows.append({
            "schedule_name": schedule_name,
            "schedule_type": schedule_type,
            "trial": trial,
            "schedule_fraction": frac,
            "n_eval": len(idx),
            "fraction_of_all_queries": len(idx) / total_queries,
            "eval_time_seconds": eval_time,
            **metrics,
        })
    return pd.DataFrame(rows)


def stop_at_target(curve_df, target_fraction=0.95):
    target = target_fraction * full_balanced_accuracy
    for _, row in curve_df.iterrows():
        if row["balanced_accuracy"] >= target:
            return {
                "stop_reason": f"target_{target_fraction:.2f}",
                "target_balanced_accuracy": target,
                "stop_n_eval": int(row["n_eval"]),
                "stop_fraction_of_all_queries": float(row["fraction_of_all_queries"]),
                "stop_balanced_accuracy": float(row["balanced_accuracy"]),
                "stop_eval_time_seconds": float(row["eval_time_seconds"]),
                "stop_lane_coverage_fraction": float(row["lane_coverage_fraction"]),
            }
    last = curve_df.iloc[-1]
    return {
        "stop_reason": f"target_{target_fraction:.2f}_not_reached",
        "target_balanced_accuracy": target,
        "stop_n_eval": int(last["n_eval"]),
        "stop_fraction_of_all_queries": float(last["fraction_of_all_queries"]),
        "stop_balanced_accuracy": float(last["balanced_accuracy"]),
        "stop_eval_time_seconds": float(last["eval_time_seconds"]),
        "stop_lane_coverage_fraction": float(last["lane_coverage_fraction"]),
    }


def stop_at_stability(curve_df, window=3, tolerance=0.01):
    vals = curve_df["balanced_accuracy"].to_numpy()
    for i in range(window, len(vals)):
        recent = vals[i-window:i+1]
        if np.max(recent) - np.min(recent) <= tolerance:
            row = curve_df.iloc[i]
            return {
                "stop_reason": f"stable_w{window}_tol{tolerance}",
                "target_balanced_accuracy": np.nan,
                "stop_n_eval": int(row["n_eval"]),
                "stop_fraction_of_all_queries": float(row["fraction_of_all_queries"]),
                "stop_balanced_accuracy": float(row["balanced_accuracy"]),
                "stop_eval_time_seconds": float(row["eval_time_seconds"]),
                "stop_lane_coverage_fraction": float(row["lane_coverage_fraction"]),
            }
    last = curve_df.iloc[-1]
    return {
        "stop_reason": f"stable_w{window}_tol{tolerance}_not_reached",
        "target_balanced_accuracy": np.nan,
        "stop_n_eval": int(last["n_eval"]),
        "stop_fraction_of_all_queries": float(last["fraction_of_all_queries"]),
        "stop_balanced_accuracy": float(last["balanced_accuracy"]),
        "stop_eval_time_seconds": float(last["eval_time_seconds"]),
        "stop_lane_coverage_fraction": float(last["lane_coverage_fraction"]),
    }


## 5. Run all schedules

In [ ]:
curve_rows = []
stop_rows = []

schedule_types = {
    "mod30": "deterministic_mod30",
    "global_low_conf": "global_low_confidence",
    "global_high_conf": "global_high_confidence",
    "hybrid_lane_low_conf": "hybrid_lane_low_confidence",
    "hybrid_lane_high_conf": "hybrid_lane_high_confidence",
    "hybrid_lane_round_robin": "hybrid_lane_round_robin",
}

for name, idx in schedules.items():
    curve = progressive_curve(idx, fractions, schedule_name=name, schedule_type=schedule_types[name], trial=-1)
    curve_rows.append(curve)

    for rule_name, stop_result in [
        ("target_95", stop_at_target(curve, target_fraction=0.95)),
        ("target_99", stop_at_target(curve, target_fraction=0.99)),
        ("stability", stop_at_stability(curve, window=3, tolerance=0.01)),
    ]:
        stop_rows.append({
            "schedule_name": name,
            "schedule_type": schedule_types[name],
            "trial": -1,
            "rule": rule_name,
            **stop_result,
        })

# Random matched baseline with same final query count as mod30.
for trial in range(N_RANDOM_TRIALS):
    ridx = random_schedule(n_keep, seed=RANDOM_STATE + 11000 + trial * 1009)
    curve = progressive_curve(ridx, fractions, schedule_name="random_matched", schedule_type="random_matched", trial=trial)
    curve_rows.append(curve)

    for rule_name, stop_result in [
        ("target_95", stop_at_target(curve, target_fraction=0.95)),
        ("target_99", stop_at_target(curve, target_fraction=0.99)),
        ("stability", stop_at_stability(curve, window=3, tolerance=0.01)),
    ]:
        stop_rows.append({
            "schedule_name": "random_matched",
            "schedule_type": "random_matched",
            "trial": trial,
            "rule": rule_name,
            **stop_result,
        })

curves_df = pd.concat(curve_rows, ignore_index=True)
stops_df = pd.DataFrame(stop_rows)

curves_csv = data_dir / "11_hybrid_modwheel_confidence_curves.csv"
stops_csv = data_dir / "11_hybrid_modwheel_confidence_stops.csv"
curves_df.to_csv(curves_csv, index=False)
stops_df.to_csv(stops_csv, index=False)

curves_df.head(), stops_df.head()


## 6. Summarize vs random

In [ ]:
random_stop_summary = (
    stops_df[stops_df["schedule_type"] == "random_matched"]
    .groupby("rule")
    .agg(
        random_stop_fraction_mean=("stop_fraction_of_all_queries", "mean"),
        random_stop_fraction_std=("stop_fraction_of_all_queries", "std"),
        random_stop_bal_acc_mean=("stop_balanced_accuracy", "mean"),
        random_stop_bal_acc_std=("stop_balanced_accuracy", "std"),
        random_stop_lane_coverage_mean=("stop_lane_coverage_fraction", "mean"),
        random_stop_lane_coverage_std=("stop_lane_coverage_fraction", "std"),
        random_stop_n_eval_mean=("stop_n_eval", "mean"),
        random_stop_n_eval_std=("stop_n_eval", "std"),
    )
    .reset_index()
)

nonrandom = stops_df[stops_df["schedule_type"] != "random_matched"][[
    "schedule_name", "schedule_type", "rule",
    "stop_fraction_of_all_queries", "stop_balanced_accuracy",
    "stop_lane_coverage_fraction", "stop_n_eval", "stop_reason"
]].rename(columns={
    "stop_fraction_of_all_queries": "stop_fraction",
    "stop_balanced_accuracy": "stop_bal_acc",
    "stop_lane_coverage_fraction": "stop_lane_coverage",
})

summary_df = nonrandom.merge(random_stop_summary, on="rule")
summary_df["delta_stop_fraction_vs_random"] = summary_df["stop_fraction"] - summary_df["random_stop_fraction_mean"]
summary_df["delta_stop_bal_acc_vs_random"] = summary_df["stop_bal_acc"] - summary_df["random_stop_bal_acc_mean"]
summary_df["delta_lane_coverage_vs_random"] = summary_df["stop_lane_coverage"] - summary_df["random_stop_lane_coverage_mean"]
summary_df["delta_stop_n_eval_vs_random"] = summary_df["stop_n_eval"] - summary_df["random_stop_n_eval_mean"]

summary_csv = data_dir / "11_hybrid_modwheel_confidence_summary.csv"
summary_df.to_csv(summary_csv, index=False)
summary_df


## 7. Figure 11a — progressive accuracy curves

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))

plot_schedules = [
    ("deterministic_mod30", "mod30"),
    ("global_low_confidence", "global low-conf"),
    ("hybrid_lane_low_confidence", "hybrid lanes + low-conf"),
    ("hybrid_lane_round_robin", "hybrid lane round-robin"),
]

for stype, label in plot_schedules:
    df = curves_df[curves_df["schedule_type"] == stype].sort_values("fraction_of_all_queries")
    ax.plot(df["fraction_of_all_queries"], df["balanced_accuracy"], marker="o", linewidth=2, label=label)

random_mean = (
    curves_df[curves_df["schedule_type"] == "random_matched"]
    .groupby("fraction_of_all_queries")["balanced_accuracy"]
    .mean()
    .reset_index()
)
random_std = (
    curves_df[curves_df["schedule_type"] == "random_matched"]
    .groupby("fraction_of_all_queries")["balanced_accuracy"]
    .std()
    .reset_index()
)

ax.plot(random_mean["fraction_of_all_queries"], random_mean["balanced_accuracy"], linewidth=2, label="random mean")
ax.fill_between(
    random_mean["fraction_of_all_queries"],
    random_mean["balanced_accuracy"] - random_std["balanced_accuracy"],
    random_mean["balanced_accuracy"] + random_std["balanced_accuracy"],
    alpha=0.2,
)

ax.axhline(full_balanced_accuracy, linestyle="--", linewidth=1, label="full readout")
ax.axhline(0.95 * full_balanced_accuracy, linestyle=":", linewidth=1, label="95% target")
ax.set_title("Hybrid Modwheel + Confidence Readout Scheduling")
ax.set_xlabel("Fraction of all test queries evaluated")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()

fig_a_svg = fig_dir / "figure_11a_hybrid_accuracy_curves.svg"
fig_a_png = fig_dir / "figure_11a_hybrid_accuracy_curves.png"
fig.savefig(fig_a_svg)
fig.savefig(fig_a_png, dpi=220)
plt.show()
fig_a_svg, fig_a_png


## 8. Figure 11b — stop fraction at 95% target

In [ ]:
target_rule = "target_95"
plot_summary = summary_df[summary_df["rule"] == target_rule].copy()
order_names = ["mod30", "global_low_conf", "hybrid_lane_low_conf", "hybrid_lane_round_robin"]
plot_summary = plot_summary.set_index("schedule_name").loc[order_names].reset_index()

labels = ["mod30", "global low-conf", "hybrid low-conf", "hybrid round-robin"]
x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.bar(x, plot_summary["stop_fraction"])
ax.axhline(plot_summary["random_stop_fraction_mean"].iloc[0], linestyle="--", linewidth=1, label="random mean")
ax.set_title("Stop Fraction to Reach 95% of Full Balanced Accuracy")
ax.set_xlabel("Schedule")
ax.set_ylabel("Fraction of all queries evaluated")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.grid(True, axis="y", alpha=0.35)
ax.legend()

for i, value in enumerate(plot_summary["stop_fraction"]):
    ax.text(i, value, f"{value:.3f}", ha="center", va="bottom")

fig.tight_layout()
fig_b_svg = fig_dir / "figure_11b_hybrid_stop_fraction_95.svg"
fig_b_png = fig_dir / "figure_11b_hybrid_stop_fraction_95.png"
fig.savefig(fig_b_svg)
fig.savefig(fig_b_png, dpi=220)
plt.show()
fig_b_svg, fig_b_png


## 9. Figure 11c — lane coverage

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))

for stype, label in plot_schedules:
    df = curves_df[curves_df["schedule_type"] == stype].sort_values("fraction_of_all_queries")
    ax.plot(df["fraction_of_all_queries"], df["lane_coverage_fraction"], marker="o", linewidth=2, label=label)

random_lane_mean = (
    curves_df[curves_df["schedule_type"] == "random_matched"]
    .groupby("fraction_of_all_queries")["lane_coverage_fraction"]
    .mean()
    .reset_index()
)

ax.plot(random_lane_mean["fraction_of_all_queries"], random_lane_mean["lane_coverage_fraction"], linewidth=2, label="random mean")
ax.set_title("Mod30 Lane Coverage During Readout")
ax.set_xlabel("Fraction of all test queries evaluated")
ax.set_ylabel("Lane coverage fraction")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()

fig_c_svg = fig_dir / "figure_11c_lane_coverage.svg"
fig_c_png = fig_dir / "figure_11c_lane_coverage.png"
fig.savefig(fig_c_svg)
fig.savefig(fig_c_png, dpi=220)
plt.show()
fig_c_svg, fig_c_png


## 10. Figure 11d — class-balance shift

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))

for stype, label in plot_schedules:
    df = curves_df[curves_df["schedule_type"] == stype].sort_values("fraction_of_all_queries")
    ax.plot(df["fraction_of_all_queries"], df["class_balance_l1_shift"], marker="o", linewidth=2, label=label)

random_balance_mean = (
    curves_df[curves_df["schedule_type"] == "random_matched"]
    .groupby("fraction_of_all_queries")["class_balance_l1_shift"]
    .mean()
    .reset_index()
)

ax.plot(random_balance_mean["fraction_of_all_queries"], random_balance_mean["class_balance_l1_shift"], linewidth=2, label="random mean")
ax.set_title("Class-Balance Shift During Readout")
ax.set_xlabel("Fraction of all test queries evaluated")
ax.set_ylabel("L1 shift vs full test distribution")
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()

fig_d_svg = fig_dir / "figure_11d_class_balance_shift.svg"
fig_d_png = fig_dir / "figure_11d_class_balance_shift.png"
fig.savefig(fig_d_svg)
fig.savefig(fig_d_png, dpi=220)
plt.show()
fig_d_svg, fig_d_png


## 11. Figure 11e — delta from random stop fraction

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.axhline(0, linestyle="--", linewidth=1)
ax.bar(labels, plot_summary["delta_stop_fraction_vs_random"])
ax.set_title("Stop Fraction Minus Random Mean (95% Target)")
ax.set_xlabel("Schedule")
ax.set_ylabel("Δ stop fraction")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.grid(True, axis="y", alpha=0.35)

for i, value in enumerate(plot_summary["delta_stop_fraction_vs_random"]):
    ax.text(i, value, f"{value:+.3f}", ha="center", va="bottom" if value >= 0 else "top")

fig.tight_layout()
fig_e_svg = fig_dir / "figure_11e_stop_fraction_delta_vs_random.svg"
fig_e_png = fig_dir / "figure_11e_stop_fraction_delta_vs_random.png"
fig.savefig(fig_e_svg)
fig.savefig(fig_e_png, dpi=220)
plt.show()
fig_e_svg, fig_e_png


## 12. Interpretation helper

In [ ]:
print("Full balanced accuracy:", full_balanced_accuracy)
print("95% target:", 0.95 * full_balanced_accuracy)
print()
display(summary_df)

for _, row in plot_summary.iterrows():
    print(
        f"{row['schedule_name']}: stop_fraction={row['stop_fraction']:.3f}, "
        f"random_mean={row['random_stop_fraction_mean']:.3f}, "
        f"delta={row['delta_stop_fraction_vs_random']:+.3f}, "
        f"lane_coverage={row['stop_lane_coverage']:.3f}"
    )

print("""
Notebook 11 interpretation:

- Notebook 10 tested global confidence-aware scheduling.
- Notebook 11 tests hybrid scheduling: mod30 lane structure + confidence ranking.
- Lane coverage measures whether a schedule spreads evaluation across modular lanes.
- If hybrid stop fraction is lower than random, it reaches target behavior with fewer queries.
- If hybrid matches random, value is reproducible lane coverage and structured readout.
- If hybrid underperforms, modular coverage may trade speed for distributional coverage.

Guardrail:
This is readout scheduling, not a claim about QOS internals, quantum advantage, or model improvement.
""")


## 13. Download outputs

In [ ]:
from google.colab import files

for path in [
    "data/11_hybrid_modwheel_confidence_curves.csv",
    "data/11_hybrid_modwheel_confidence_stops.csv",
    "data/11_hybrid_modwheel_confidence_summary.csv",
    "figures/figure_11a_hybrid_accuracy_curves.svg",
    "figures/figure_11b_hybrid_stop_fraction_95.svg",
    "figures/figure_11c_lane_coverage.svg",
    "figures/figure_11d_class_balance_shift.svg",
    "figures/figure_11e_stop_fraction_delta_vs_random.svg",
]:
    files.download(path)
